# 从零实现 Graphormer 风格图 Transformer：空间偏置、Padding Mask 与图 Token

本 Notebook 只用 PyTorch 基础张量手写 manual Q/K/V 多头自注意力、Graphormer 风格节点/degree embedding、最短路 spatial bias、encoder block 与图级分类器；不调用 `nn.MultiheadAttention`、`nn.Transformer`、PyG 或 DGL。

可执行合同覆盖：缩放与稳定 softmax 数值 oracle、多图 padded batch、全 mask fail-closed、padding 不变性、节点重排时同步距离矩阵、不连通 bucket、受控训练、梯度与制品绑定。固定 seed、CPU、单线程、离线小图；教学结果不等价于大规模 Graphormer 复现实验。

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

from dataclasses import dataclass
import hashlib
import json
import math
import random
import time

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 3901
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
DTYPE = torch.float32

def canonical_hash(payload) -> str:
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]

assert DEVICE.type == "cpu" and torch.get_num_threads() == 1
assert torch.initial_seed() == SEED
assert not any(name in globals() for name in ("torch_geometric", "dgl"))

## 1. 受控图分类与切分

类别 0 为环、类别 1 为星形，节点数 5–8。节点只有离散类型（按局部编号奇偶交替），主要结构信号来自 degree 与 shortest-path distance。每类 18 张图，按图 ID 划分为 12 train、3 validation、3 test；任何节点级展开都必须继承图级 split，不能让同一张图的节点落入不同集合。

In [ ]:
@dataclass(frozen=True)
class GraphItem:
    graph_id: str
    node_type: torch.Tensor
    edge_pairs: tuple[tuple[int, int], ...]
    label: int
    split: str

def make_graph(label: int, index: int, split: str) -> GraphItem:
    if label not in (0, 1): raise ValueError("label 非法")
    n = 5 + index % 4
    if label == 0:
        pairs = {tuple(sorted((i, (i + 1) % n))) for i in range(n)}
    else:
        pairs = {(0, i) for i in range(1, n)}
    node_type = torch.tensor([1 + (i % 2) for i in range(n)], dtype=torch.long)
    return GraphItem(f"{split}-c{label}-{index:02d}", node_type, tuple(sorted(pairs)), label, split)

graphs = []
for label in (0, 1):
    for i in range(18):
        split = "train" if i < 12 else ("val" if i < 15 else "test")
        graphs.append(make_graph(label, i, split))
split_graphs = {s: [g for g in graphs if g.split == s] for s in ("train", "val", "test")}
assert tuple(len(split_graphs[s]) for s in ("train", "val", "test")) == (24, 6, 6)
assert len({g.graph_id for g in graphs}) == 36
assert all(g.node_type.ndim == 1 and g.node_type.numel() >= 5 for g in graphs)
assert not ({g.graph_id for g in split_graphs["train"]} & {g.graph_id for g in split_graphs["test"]})

## 2. 最短路 spatial bucket 与不连通语义

Graphormer 把节点对最短路距离 $d(i,j)$ 映射为每个 attention head 的偏置 $b_{d(i,j)}^{(h)}$，再加到 QK logit。为限制词表，距离超过 `max_distance` 时截断；不可达节点对使用独立 `disconnected=max_distance+1` bucket，不能与“很远但可达”混在一起。

教学实现用 Floyd–Warshall，复杂度 $O(N^3)$，只适合小图和 oracle。生产应离线预计算、缓存并把距离算法/截断值写入特征制品。

In [ ]:
MAX_DISTANCE = 4
DISCONNECTED_BUCKET = MAX_DISTANCE + 1
NUM_DISTANCE_BUCKETS = MAX_DISTANCE + 2

def validate_pairs(num_nodes: int, edge_pairs: tuple[tuple[int, int], ...]) -> None:
    if num_nodes <= 0: raise ValueError("空图不允许进入 Graphormer")
    seen = set()
    for u, v in edge_pairs:
        if not (0 <= u < num_nodes and 0 <= v < num_nodes) or u == v:
            raise ValueError("边端点越界或含自环")
        key = tuple(sorted((u, v)))
        if key in seen: raise ValueError("无向边重复")
        seen.add(key)

def shortest_path_buckets(num_nodes: int, edge_pairs: tuple[tuple[int, int], ...],
                          max_distance: int = MAX_DISTANCE) -> torch.Tensor:
    if not isinstance(max_distance, int) or max_distance < 0:
        raise ValueError("max_distance 必须是非负整数")
    validate_pairs(num_nodes, edge_pairs)
    inf = num_nodes + max_distance + 10
    dist = torch.full((num_nodes, num_nodes), inf, dtype=torch.long)
    dist.fill_diagonal_(0)
    for u, v in edge_pairs:
        dist[u, v] = 1; dist[v, u] = 1
    for k in range(num_nodes):
        dist = torch.minimum(dist, dist[:, k:k+1] + dist[k:k+1, :])
    disconnected = dist >= inf
    buckets = dist.clamp(max=max_distance)
    buckets[disconnected] = max_distance + 1
    return buckets

disconnected = shortest_path_buckets(4, ((0, 1), (2, 3)))
path6 = shortest_path_buckets(6, tuple((i, i + 1) for i in range(5)))
assert int(disconnected[0, 2]) == DISCONNECTED_BUCKET
assert int(disconnected[0, 1]) == 1 and torch.equal(disconnected.diag(), torch.zeros(4, dtype=torch.long))
assert int(path6[0, 5]) == MAX_DISTANCE and int(path6.max()) == MAX_DISTANCE
assert torch.equal(path6, path6.T)

try:
    shortest_path_buckets(2, ((0, 1),), max_distance=-1)
    raise AssertionError("负 max_distance 未被拒绝")
except ValueError as exc:
    assert "非负" in str(exc)


## 3. 多图 Padded Batch

Dense attention 需要把一批图补齐到最大节点数 $M$：`node_type/degree:(B,M)`、`distance:(B,M,M)`、`node_valid:(B,M)`。padding 节点固定使用 `node_type=0, degree=0`，任一轴涉及 padding 的 distance 固定为 disconnected sentinel；这些张量与 graph ID 会共同生成 batch snapshot hash。模型再在每张图前插入一个 graph token，因此序列长为 $L=M+1$，graph token 永远有效。

`node_valid` 与 distance 必须由同一图快照生成，模型入口会重算 snapshot hash，并验证有效距离矩阵的对称性与零对角。若只 mask key、不处理 padded query，虽然 graph token 可能不受影响，padding 表征仍会产生无意义激活并污染后续读出；本实现每个 block 后显式归零无效 query。

In [ ]:
@dataclass(frozen=True)
class DenseGraphBatch:
    node_type: torch.Tensor
    degree: torch.Tensor
    distance: torch.Tensor
    node_valid: torch.Tensor
    labels: torch.Tensor
    graph_ids: tuple[str, ...]
    snapshot_hash: str


def dense_snapshot_hash(node_type: torch.Tensor, degree: torch.Tensor, distance: torch.Tensor,
                        node_valid: torch.Tensor, graph_ids: tuple[str, ...]) -> str:
    payload = {"node_type": node_type.cpu().tolist(), "degree": degree.cpu().tolist(),
               "distance": distance.cpu().tolist(), "node_valid": node_valid.cpu().tolist(),
               "graph_ids": list(graph_ids),
               "distance_contract": {"max": MAX_DISTANCE, "disconnected": DISCONNECTED_BUCKET}}
    return canonical_hash(payload)


def graph_degree(graph: GraphItem) -> torch.Tensor:
    validate_pairs(graph.node_type.numel(), graph.edge_pairs)
    degree = torch.zeros(graph.node_type.numel(), dtype=torch.long)
    for u, v in graph.edge_pairs:
        degree[u] += 1; degree[v] += 1
    return degree


def pad_graphs(items: list[GraphItem]) -> DenseGraphBatch:
    if not items: raise ValueError("至少需要一张图")
    if len({g.graph_id for g in items}) != len(items): raise ValueError("batch 内 graph_id 重复")
    max_nodes = max(int(g.node_type.numel()) for g in items)
    bsz = len(items)
    node_type = torch.zeros((bsz, max_nodes), dtype=torch.long)
    degree = torch.zeros((bsz, max_nodes), dtype=torch.long)
    distance = torch.full((bsz, max_nodes, max_nodes), DISCONNECTED_BUCKET, dtype=torch.long)
    valid = torch.zeros((bsz, max_nodes), dtype=torch.bool)
    for i, graph in enumerate(items):
        n = int(graph.node_type.numel()); validate_pairs(n, graph.edge_pairs)
        node_type[i, :n] = graph.node_type
        degree[i, :n] = graph_degree(graph)
        distance[i, :n, :n] = shortest_path_buckets(n, graph.edge_pairs)
        valid[i, :n] = True
    labels = torch.tensor([g.label for g in items], dtype=torch.long)
    ids = tuple(g.graph_id for g in items)
    snapshot_hash = dense_snapshot_hash(node_type, degree, distance, valid, ids)
    return DenseGraphBatch(node_type, degree, distance, valid, labels, ids, snapshot_hash)


dense_probe = pad_graphs([graphs[0], graphs[15]])
assert dense_probe.node_type.shape == dense_probe.degree.shape == dense_probe.node_valid.shape == (2, 8)
assert dense_probe.distance.shape == (2, 8, 8)
assert torch.equal(dense_probe.node_valid.sum(1), torch.tensor([5, 8]))
assert torch.all(dense_probe.distance[0, 5:, :] == DISCONNECTED_BUCKET)
assert dense_probe.snapshot_hash == dense_snapshot_hash(
    dense_probe.node_type, dense_probe.degree, dense_probe.distance,
    dense_probe.node_valid, dense_probe.graph_ids)
try:
    pad_graphs([])
    raise AssertionError("空 batch 未被拒绝")
except ValueError as exc:
    assert "至少需要" in str(exc)


## 4. 手写缩放点积注意力与数值 oracle

对每个 head：

$$A=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_h}}+B_{spatial}+M_{key}\right),\qquad O=AV.$$

`Q/K/V:(B,H,L,d_h)`，偏置 `(B,H,L,L)`。除以 $\sqrt{d_h}$ 防止维度增大时 logit 方差过大；softmax 前减行最大值（PyTorch softmax 内部稳定实现）避免指数溢出。若某个样本所有 key 都被 mask，softmax 没有定义，必须显式报错。

In [ ]:
def scaled_masked_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                            bias: torch.Tensor, valid: torch.Tensor):
    if q.ndim != 4 or not (q.shape == k.shape == v.shape):
        raise ValueError("q/k/v 必须同 shape (B,H,L,D)")
    if q.shape[-1] <= 0 or not all(torch.isfinite(tensor).all() for tensor in (q, k, v, bias)):
        raise ValueError("q/k/v/bias 必须是非空有限张量")
    bsz, heads, length, head_dim = q.shape
    if bias.shape != (bsz, heads, length, length) or valid.shape != (bsz, length):
        raise ValueError("bias 或 valid shape 不匹配")
    if not valid.any(dim=1).all():
        raise ValueError("存在全 mask 样本，注意力无定义")
    logits = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(head_dim) + bias
    logits = logits.masked_fill(~valid[:, None, None, :], -torch.inf)
    attention = torch.softmax(logits, dim=-1)
    attention = attention * valid[:, None, :, None].to(attention.dtype)
    out = torch.matmul(attention, v)
    out = out * valid[:, None, :, None].to(out.dtype)
    if not torch.isfinite(attention).all() or not torch.isfinite(out).all():
        raise FloatingPointError("attention 或输出出现非有限值")
    return out, attention

q = torch.zeros((1, 1, 3, 2)); k = torch.zeros_like(q)
v = torch.tensor([[[[1., 3.], [5., 7.], [100., 100.]]]])
valid = torch.tensor([[True, True, False]])
mean_out, mean_attn = scaled_masked_attention(q, k, v, torch.zeros(1, 1, 3, 3), valid)
assert torch.allclose(mean_out[0, 0, 0], torch.tensor([3., 5.]), atol=1e-7)
assert torch.equal(mean_out[0, 0, 2], torch.zeros(2))
assert torch.allclose(mean_attn[0, 0, 0], torch.tensor([0.5, 0.5, 0.0]), atol=1e-7)

q2 = torch.tensor([[[[1., 0.], [0., 0.]]]])
k2 = torch.tensor([[[[1., 0.], [0., 1.]]]])
v2 = torch.eye(2).reshape(1, 1, 2, 2)
scaled_out, scaled_attn = scaled_masked_attention(q2, k2, v2, torch.zeros(1, 1, 2, 2), torch.ones(1, 2, dtype=torch.bool))
expected = torch.softmax(torch.tensor([1 / math.sqrt(2), 0.0]), dim=0)
assert torch.allclose(scaled_attn[0, 0, 0], expected, atol=1e-7)
assert torch.isfinite(scaled_out).all()
try:
    scaled_masked_attention(q2, k2, v2, torch.zeros(1, 1, 2, 2), torch.zeros(1, 2, dtype=torch.bool))
    raise AssertionError("全 mask 注意力未被拒绝")
except ValueError as exc:
    assert "全 mask" in str(exc)

nan_bias = torch.zeros(1, 1, 2, 2); nan_bias[0, 0, 0, 0] = torch.nan
try:
    scaled_masked_attention(q2, k2, v2, nan_bias, torch.ones(1, 2, dtype=torch.bool))
    raise AssertionError("NaN spatial bias 未被拒绝")
except ValueError as exc:
    assert "有限" in str(exc)


## 5. `ManualMultiHeadAttention`

四个线性映射显式产生 Q、K、V、输出；`hidden_dim` 必须能被 `num_heads` 整除。每个 head 只看到自己的 $d_h$ 维子空间，但共享同一个 key/query 有效性 mask，并接收不同的 spatial bias。返回 attention 便于审计，而不是只返回无法解释的图向量。

In [ ]:
class ManualMultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int):
        super().__init__()
        if hidden_dim <= 0 or num_heads <= 0 or hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim 必须为 num_heads 的正整数倍")
        self.hidden_dim, self.num_heads = hidden_dim, num_heads
        self.head_dim = hidden_dim // num_heads
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

    def _split(self, x: torch.Tensor) -> torch.Tensor:
        bsz, length, _ = x.shape
        return x.view(bsz, length, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x: torch.Tensor, spatial_bias: torch.Tensor, valid: torch.Tensor):
        if x.ndim != 3 or x.shape[2] != self.hidden_dim or not torch.isfinite(x).all():
            raise ValueError("attention 输入 shape 或数值非法")
        q, k, v = self._split(self.q_proj(x)), self._split(self.k_proj(x)), self._split(self.v_proj(x))
        out, attention = scaled_masked_attention(q, k, v, spatial_bias, valid)
        out = out.transpose(1, 2).contiguous().view(x.shape)
        out = self.out_proj(out) * valid[:, :, None].to(out.dtype)
        if not torch.isfinite(out).all():
            raise FloatingPointError("attention 输出投影出现非有限值")
        return out, attention

mha_probe = ManualMultiHeadAttention(16, 4)
probe_x = torch.randn(2, 5, 16)
probe_valid = torch.tensor([[1,1,1,1,1], [1,1,1,0,0]], dtype=torch.bool)
probe_out, probe_attn = mha_probe(probe_x, torch.zeros(2, 4, 5, 5), probe_valid)
assert probe_out.shape == (2, 5, 16) and probe_attn.shape == (2, 4, 5, 5)
assert torch.equal(probe_out[1, 3:], torch.zeros(2, 16))
assert torch.allclose(probe_attn[1, :, :3, :3].sum(-1), torch.ones(4, 3), atol=1e-6)

## 6. Graphormer 风格编码器

节点初始表示为 `node type embedding + degree embedding`。在序列最前插入可学习 graph token；节点—节点 attention bias 来自最短路 bucket embedding，graph token 与节点之间使用单独的 virtual-distance bias。每个 block 使用 pre-LayerNorm：

$$X\leftarrow X+\mathrm{MHA}(\mathrm{LN}(X)),\qquad X\leftarrow X+\mathrm{FFN}(\mathrm{LN}(X)).$$

最终只读取 graph token 分类。这个实现抓住 Graphormer 的结构编码核心，但省略 edge feature、多跳 edge encoding、centrality 的有向入/出度拆分等论文完整组件，因此称“Graphormer 风格”而非逐配置复刻。

In [ ]:
class GraphEncoderBlock(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int, expansion: int = 2):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.attn = ManualMultiHeadAttention(hidden_dim, num_heads)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.ff1 = nn.Linear(hidden_dim, expansion * hidden_dim)
        self.ff2 = nn.Linear(expansion * hidden_dim, hidden_dim)

    def forward(self, x: torch.Tensor, bias: torch.Tensor, valid: torch.Tensor):
        attn_out, attention = self.attn(self.norm1(x), bias, valid)
        x = (x + attn_out) * valid[:, :, None].to(x.dtype)
        ff = self.ff2(F.gelu(self.ff1(self.norm2(x))))
        x = (x + ff) * valid[:, :, None].to(x.dtype)
        return x, attention

class GraphormerClassifier(nn.Module):
    def __init__(self, num_node_types: int, max_degree: int, hidden_dim: int,
                 num_heads: int, num_layers: int, num_classes: int):
        super().__init__()
        if min(num_node_types, max_degree, hidden_dim, num_heads, num_layers, num_classes) <= 0:
            raise ValueError("模型配置必须为正")
        self.num_node_types = num_node_types
        self.max_degree = max_degree
        self.num_heads = num_heads
        self.node_embedding = nn.Embedding(num_node_types, hidden_dim)
        self.degree_embedding = nn.Embedding(max_degree + 1, hidden_dim)
        self.spatial_embedding = nn.Embedding(NUM_DISTANCE_BUCKETS, num_heads)
        self.graph_token = nn.Parameter(torch.zeros(1, 1, hidden_dim))
        self.virtual_distance = nn.Parameter(torch.zeros(num_heads))
        self.blocks = nn.ModuleList([GraphEncoderBlock(hidden_dim, num_heads) for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)
        nn.init.normal_(self.graph_token, std=0.02)

    def build_inputs(self, batch: DenseGraphBatch):
        if batch.node_type.shape != batch.degree.shape or batch.node_type.shape != batch.node_valid.shape:
            raise ValueError("node_type/degree/valid shape 不一致")
        if batch.node_type.dtype != torch.long or batch.degree.dtype != torch.long or batch.distance.dtype != torch.long:
            raise ValueError("node_type/degree/distance 必须是 long")
        if batch.node_valid.dtype != torch.bool:
            raise ValueError("node_valid 必须是 bool")
        bsz, max_nodes = batch.node_type.shape
        if batch.distance.shape != (bsz, max_nodes, max_nodes):
            raise ValueError("distance shape 不匹配")
        if len(batch.graph_ids) != bsz or len(set(batch.graph_ids)) != bsz:
            raise ValueError("graph_ids 数量不匹配或重复")
        expected_snapshot = dense_snapshot_hash(batch.node_type, batch.degree, batch.distance,
                                                batch.node_valid, batch.graph_ids)
        if batch.snapshot_hash != expected_snapshot:
            raise ValueError("dense batch snapshot 指纹不匹配")
        if not batch.node_valid.any(dim=1).all():
            raise ValueError("存在空图")
        for graph_id in range(bsz):
            valid = batch.node_valid[graph_id]
            valid_distance = batch.distance[graph_id][valid][:, valid]
            if not torch.equal(valid_distance, valid_distance.T):
                raise ValueError("有效 distance 必须对称")
            if not torch.equal(valid_distance.diag(), torch.zeros(valid_distance.shape[0], dtype=torch.long)):
                raise ValueError("有效 distance 对角线必须为 0")
            padding_pair = ~(valid[:, None] & valid[None, :])
            if bool(padding_pair.any()) and not torch.all(batch.distance[graph_id][padding_pair] == DISCONNECTED_BUCKET):
                raise ValueError("padding distance 必须使用 disconnected sentinel")
        if bool((batch.node_type[~batch.node_valid] != 0).any()) or bool((batch.degree[~batch.node_valid] != 0).any()):
            raise ValueError("padding node_type/degree 必须为 0")
        valid_types = batch.node_type[batch.node_valid]
        if valid_types.numel() == 0 or int(valid_types.min()) < 0 or int(valid_types.max()) >= self.num_node_types:
            raise ValueError("node type 超出冻结词表")
        if int(batch.degree.max()) > self.max_degree or int(batch.degree.min()) < 0:
            raise ValueError("degree 超出冻结词表")
        if int(batch.distance.min()) < 0 or int(batch.distance.max()) >= NUM_DISTANCE_BUCKETS:
            raise ValueError("distance bucket 越界")
        node_x = self.node_embedding(batch.node_type) + self.degree_embedding(batch.degree)
        node_x = node_x * batch.node_valid[:, :, None].to(node_x.dtype)
        token = self.graph_token.expand(bsz, -1, -1)
        x = torch.cat([token, node_x], dim=1)
        valid = torch.cat([torch.ones(bsz, 1, dtype=torch.bool, device=x.device), batch.node_valid], dim=1)
        length = max_nodes + 1
        bias = x.new_zeros((bsz, self.num_heads, length, length))
        node_bias = self.spatial_embedding(batch.distance).permute(0, 3, 1, 2)
        bias[:, :, 1:, 1:] = node_bias
        bias[:, :, 0, 1:] = self.virtual_distance[None, :, None]
        bias[:, :, 1:, 0] = self.virtual_distance[None, :, None]
        return x, bias, valid

    def forward(self, batch: DenseGraphBatch, return_attention: bool = False):
        x, bias, valid = self.build_inputs(batch)
        attentions = []
        for block in self.blocks:
            x, attention = block(x, bias, valid)
            attentions.append(attention)
        logits = self.classifier(self.final_norm(x[:, 0]))
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Graphormer logits 出现非有限值")
        return (logits, attentions) if return_attention else logits

torch.manual_seed(SEED + 1)
model_probe = GraphormerClassifier(3, 8, 24, 4, 2, 2).eval()
logits_probe, attn_probe = model_probe(dense_probe, return_attention=True)
assert logits_probe.shape == (2, 2) and len(attn_probe) == 2
assert attn_probe[0].shape == (2, 4, 9, 9)
assert torch.isfinite(logits_probe).all() and torch.isfinite(attn_probe[0]).all()
bad_node_type = dense_probe.node_type.clone(); bad_node_type[0, 0] = 3
bad_vocab_hash = dense_snapshot_hash(bad_node_type, dense_probe.degree, dense_probe.distance,
                                      dense_probe.node_valid, dense_probe.graph_ids)
bad_vocab_batch = DenseGraphBatch(bad_node_type, dense_probe.degree, dense_probe.distance,
                                  dense_probe.node_valid, dense_probe.labels,
                                  dense_probe.graph_ids, bad_vocab_hash)
try:
    model_probe(bad_vocab_batch)
    raise AssertionError("越界 node type 未被拒绝")
except ValueError as exc:
    assert "node type" in str(exc)

## 7. Padding 与节点置换不变性

两个关键 oracle：

1. 同一张小图单独推理，或与更大图一起 padding，logits 应一致；否则 pad token 泄漏进 attention。
2. 节点重排时，`node_type`、degree 和距离矩阵必须按同一置换同步变化，graph-token logits 应一致。

第二点尤其重要：只重排节点特征而不重排距离矩阵，是把错误结构偏置绑定到了节点，不是在测试置换不变性。

In [ ]:
small, large = graphs[0], graphs[17]
single_batch = pad_graphs([small])
mixed_batch = pad_graphs([small, large])
model_probe.eval()
with torch.no_grad():
    single_logits = model_probe(single_batch)
    mixed_logits = model_probe(mixed_batch)[:1]
assert torch.allclose(single_logits, mixed_logits, atol=2e-6)

original = pad_graphs([graphs[3]])
n = int(original.node_valid.sum())
perm = torch.tensor([2, 0, 4, 1, 3, 5, 6, 7][:n])
perm_node_type = original.node_type[:, perm]
perm_degree = original.degree[:, perm]
perm_distance = original.distance[:, perm][:, :, perm]
perm_valid = original.node_valid[:, perm]
perm_ids = ("permuted",)
perm_hash = dense_snapshot_hash(perm_node_type, perm_degree, perm_distance, perm_valid, perm_ids)
permuted = DenseGraphBatch(perm_node_type, perm_degree, perm_distance, perm_valid,
                           original.labels, perm_ids, perm_hash)
with torch.no_grad():
    original_logits = model_probe(original)
    permuted_logits = model_probe(permuted)
assert torch.allclose(original_logits, permuted_logits, atol=2e-6)
assert not torch.equal(original.node_type, original.node_type[:, perm])

# 任一缓存张量被改动但 snapshot 未同步，都必须在模型入口失败。
tampered_distance = dense_probe.distance.clone()
tampered_distance[0, 0, 1] = 2
tampered_batch = DenseGraphBatch(dense_probe.node_type, dense_probe.degree, tampered_distance,
                                 dense_probe.node_valid, dense_probe.labels,
                                 dense_probe.graph_ids, dense_probe.snapshot_hash)
try:
    model_probe(tampered_batch)
    raise AssertionError("被篡改 distance 与旧 snapshot 一起进入模型")
except ValueError as exc:
    assert "snapshot" in str(exc)

# 即便攻击者重算指纹，非对称 distance 仍会被结构合同拒绝。
asymmetric_distance = dense_probe.distance.clone()
asymmetric_distance[0, 0, 1] = 1
asymmetric_distance[0, 1, 0] = 2
asymmetric_hash = dense_snapshot_hash(dense_probe.node_type, dense_probe.degree, asymmetric_distance,
                                      dense_probe.node_valid, dense_probe.graph_ids)
asymmetric_batch = DenseGraphBatch(dense_probe.node_type, dense_probe.degree, asymmetric_distance,
                                   dense_probe.node_valid, dense_probe.labels,
                                   dense_probe.graph_ids, asymmetric_hash)
try:
    model_probe(asymmetric_batch)
    raise AssertionError("非对称 distance 未被拒绝")
except ValueError as exc:
    assert "对称" in str(exc)


## 8. 受控训练、梯度与 checkpoint

全批训练 24 张图，validation 每 5 轮选择 checkpoint，test 冻结后打开一次。我们检查 spatial embedding、Q projection、graph token 和分类头都获得非零有限梯度，证明结构偏置、attention 与读出链路都参与优化。

In [ ]:
train_batch = pad_graphs(split_graphs["train"])
val_batch = pad_graphs(split_graphs["val"])
test_batch = pad_graphs(split_graphs["test"])

torch.manual_seed(SEED + 2)
model = GraphormerClassifier(3, 8, 32, 4, 2, 2).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.012, weight_decay=1e-4)
initial_loss, best_val, best_state = None, -1.0, None
train_start = time.perf_counter()
for epoch in range(121):
    model.train(); optimizer.zero_grad(set_to_none=True)
    logits = model(train_batch)
    loss = F.cross_entropy(logits, train_batch.labels)
    if initial_loss is None: initial_loss = float(loss.detach())
    loss.backward()
    if epoch == 0:
        grads = [model.spatial_embedding.weight.grad, model.blocks[0].attn.q_proj.weight.grad,
                 model.graph_token.grad, model.classifier.weight.grad]
        assert all(g is not None and torch.isfinite(g).all() and float(g.abs().sum()) > 0 for g in grads)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    optimizer.step()
    if epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            val_acc = float((model(val_batch).argmax(1) == val_batch.labels).float().mean())
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

assert initial_loss is not None and best_state is not None
model.load_state_dict(best_state); model.eval()
with torch.no_grad():
    final_train_loss = float(F.cross_entropy(model(train_batch), train_batch.labels))
    test_acc = float((model(test_batch).argmax(1) == test_batch.labels).float().mean())
train_seconds = time.perf_counter() - train_start
assert final_train_loss < initial_loss * 0.4
assert best_val >= 0.99 and test_acc >= 0.99
assert train_seconds < 12.0
print({"initial_loss": round(initial_loss, 4), "final_loss": round(final_train_loss, 4),
       "val_acc": best_val, "test_acc": test_acc, "seconds": round(train_seconds, 3)})

## 9. 失败模式、复杂度与生产边界

- **全 mask 行**：用 `-inf` 后 softmax 会产生 NaN；空图/全 mask 必须在 attention 前拒绝。
- **只 mask key**：还应归零 padded query，防止后续 pooling 或残差误用其表征。
- **距离矩阵不同步**：节点重编号后必须对 distance 的两个轴同时置换；node/degree/distance/valid/graph ID 共同绑定 batch snapshot 指纹。
- **不可达与远距离混桶**：语义不同，应有独立 disconnected bucket，并监控其占比漂移。
- **degree 词表溢出**：不能默默 clamp 新图的超大度数；要么显式 OOV bucket，要么拒绝并升级制品。
- **二次方成本**：单层 dense attention 为 $O(BL^2D)$ 时间与 $O(BHL^2)$ attention 内存。大图需子图、稀疏 attention 或分层 pooling，且要重新验证结构偏置语义。

权限过滤、图快照生成、距离预计算必须在同一可审计边界；先对全局图算距离再裁剪可见节点，仍可能泄漏隐藏结构。

In [ ]:
def state_hash(module: nn.Module) -> str:
    digest = hashlib.sha256()
    for name, tensor in sorted(module.state_dict().items()):
        digest.update(name.encode("utf-8")); digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()[:20]

snapshot = [{"id": g.graph_id, "types": g.node_type.tolist(), "edges": list(g.edge_pairs),
             "label": g.label, "split": g.split} for g in graphs]
artifact = {
    "architecture": "GraphormerStyle-2layer-4head-v1",
    "seed": SEED,
    "node_vocab_hash": canonical_hash({"PAD": 0, "local_index_even": 1, "local_index_odd": 2}),
    "distance_contract_hash": canonical_hash({"algorithm": "floyd-warshall", "max": MAX_DISTANCE,
                                               "disconnected": DISCONNECTED_BUCKET}),
    "batch_snapshot_schema_hash": canonical_hash({"fields": ["node_type", "degree", "distance",
                                                               "node_valid", "graph_ids"]}),
    "graph_snapshot_hash": canonical_hash(snapshot),
    "split_hash": canonical_hash({s: [g.graph_id for g in split_graphs[s]] for s in split_graphs}),
    "state_hash": state_hash(model),
    "metrics": {"best_val_accuracy": best_val, "test_accuracy": test_acc},
}
artifact["artifact_id"] = canonical_hash(artifact)

def predict(item: GraphItem, expected_artifact_id: str) -> int:
    if expected_artifact_id != artifact["artifact_id"]:
        raise RuntimeError("模型、距离合同或图快照绑定不一致")
    batch = pad_graphs([item])
    with torch.no_grad(): return int(model(batch).argmax(1).item())

assert predict(graphs[-1], artifact["artifact_id"]) in (0, 1)
assert len({artifact["node_vocab_hash"], artifact["distance_contract_hash"],
            artifact["batch_snapshot_schema_hash"], artifact["graph_snapshot_hash"],
            artifact["split_hash"], artifact["state_hash"]}) == 6
try:
    predict(graphs[-1], "stale-artifact")
    raise AssertionError("过期制品未被拒绝")
except RuntimeError as exc:
    assert "绑定" in str(exc)
print({"artifact_id": artifact["artifact_id"], "state_hash": artifact["state_hash"]})

## 10. 面试复盘与论文来源

一个扎实的 Graph Transformer 回答，应说明结构如何进入 attention，而不是只说“把图转成序列”：节点/degree 编码、最短路偏置、graph token、QKV shape、mask 行为、置换不变性和 $L^2$ 成本缺一不可。工程上还要绑定距离算法、bucket 词表、图快照和权重。

主要来源：Ying et al., [**Do Transformers Really Perform Bad for Graph Representation? (Graphormer)**](https://arxiv.org/abs/2106.05234), NeurIPS 2021；Vaswani et al., [**Attention Is All You Need**](https://arxiv.org/abs/1706.03762), NeurIPS 2017。这里是 Graphormer 风格的小规模复现，并非官方代码或完整训练配方。